# Phase 2: Grad-CAM for Image Branch

This notebook generates **target-specific Grad-CAM heatmaps** for the image branch of the
multimodal restaurant review quality assessment model.

| Component | Configuration |
|---|---|
| Image Backbone | Swin-B (`swin_base_patch4_window7_224`) |
| Text Backbone | PhoBERT (`vinai/phobert-base-v2`) |
| Fusion | Cross-Attention (8 heads, hidden=512) |
| XAI Method | Manual Grad-CAM via hooks (no pytorch-grad-cam dependency) |

**Key features:**
- Per-target heatmaps (5 targets: food, price, atmosphere, service, overall)
- Per-image heatmaps for multi-image reviews (up to 4 images)
- Text inputs held fixed during image explanation
- Hook-based activation/gradient isolation for faithful multi-image attribution

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone -b visualization https://github.com/lechihoang/SE365.git /content/SE365 2>/dev/null || echo 'Repo already cloned'
%cd /content/SE365
!pip install -q -r requirements.txt

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

**Edit this cell only.**

In [ ]:
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/gradcam'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

SAMPLE_INDICES = [0, 1, 2]  # Samples to process

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_OUT_DIR  : {XAI_OUT_DIR}')
print(f'Samples      : {SAMPLE_INDICES}')

### STEP 5: Imports and Seed

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 5/17 — Imports and Seed')
print('='*60)

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    IMAGE_FEATURE_DIM, DEFAULT_SEED, DEFAULT_DPI, THESIS_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, get_prediction,
    save_raw_values, get_metadata,
)
from xai.gradcam_explainer import (
    GradCAMExplainer,
    compute_gradcam_for_image,
    overlay_cam_on_image,
    find_target_layer,
    create_5target_comparison,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

print(f'Device  : {device}')
print(f'Seed    : {SEED}')
print(f'PyTorch : {torch.__version__}')
print(f'\nStep 5 done ({time.time()-t0:.1f}s)')

### STEP 6: Load Model

Loads the best model and patches PhoBERT attention to eager mode (for Phase 3 compatibility).

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 6/17 — Load Model')
print('='*60)

model, config = load_model(EXP_DIR, device=device)

# Patch sdpa -> eager attention (same as Phase 1)
encoder = model.text_model.encoder
if hasattr(encoder, 'config'):
    encoder.config._attn_implementation = 'eager'
    encoder.config.attn_implementation = 'eager'
patched = 0
try:
    from transformers.models.roberta.modeling_roberta import RobertaSelfAttention
    if hasattr(encoder, 'encoder') and hasattr(encoder.encoder, 'layer'):
        for layer_module in encoder.encoder.layer:
            attn = layer_module.attention.self
            if 'Sdpa' in type(attn).__name__ or 'Flash' in type(attn).__name__:
                eager_attn = RobertaSelfAttention(encoder.config)
                eager_attn.load_state_dict(attn.state_dict())
                eager_attn.to(next(attn.parameters()).device)
                layer_module.attention.self = eager_attn
                patched += 1
except Exception as e:
    print(f'Attention patch skipped: {e}')
if patched > 0:
    print(f'Patched {patched} attention layers: sdpa -> eager')

total_params = sum(p.numel() for p in model.parameters())
print(f'Model   : {model.__class__.__name__} ({total_params:,} params)')
print(f'\nStep 6 done ({time.time()-t0:.1f}s)')

### STEP 7: Load Tokenizer, Image Processor, and Select Data Split

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 7/17 — Tokenizer & Data Split')
print('='*60)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)

tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

# Select data split
test_csv = os.path.join(DATA_DIR, 'test.csv')
val_csv  = os.path.join(DATA_DIR, 'val.csv')
SPLIT_CSV = test_csv if os.path.isfile(test_csv) else val_csv
SPLIT_NAME = 'test' if SPLIT_CSV == test_csv else 'validation'

print(f'Split   : {SPLIT_NAME} ({SPLIT_CSV})')
print(f'\nStep 7 done ({time.time()-t0:.1f}s)')

### STEP 8: Load and Display a Sample

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 8/17 — Load Sample')
print('='*60)

demo_idx = SAMPLE_INDICES[0]
sample = load_single_sample(
    csv_path=SPLIT_CSV, idx=demo_idx,
    tokenizer=tokenizer, image_processor=image_processor,
    image_dir=IMAGE_DIR, device=device,
)

result = get_prediction(model, sample)
num_images = sample['num_real_images']

print(f'Sample idx   : {demo_idx}')
print(f'Num images   : {num_images}')
print(f'Text (80ch)  : {sample["text"][:80]}...')
print(f'\nPredictions vs Ground Truth:')
print(f'{"Target":<28s} {"Pred":>8s} {"GT":>8s}')
print('-'*46)
for name in TARGET_NAMES:
    p = result['predictions'][name]
    g = result['ground_truth'][name]
    print(f'{name:<28s} {p:8.3f} {g:8.2f}')

# Display images
fig, axes = plt.subplots(1, num_images, figsize=(4*num_images, 4))
if num_images == 1:
    axes = [axes]
for i, (img, ax) in enumerate(zip(sample['loaded_images'], axes)):
    ax.imshow(img)
    ax.set_title(f'Image {i}', fontsize=11)
    ax.axis('off')
plt.suptitle(f'Sample {demo_idx} — {num_images} image(s)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nStep 8 done ({time.time()-t0:.1f}s)')

### STEP 9: Verify Grad-CAM Target Layer

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 9/17 — Verify Target Layer')
print('='*60)

target_layer = find_target_layer(model)
print(f'Target layer : {type(target_layer).__name__}')
print(f'\nStep 9 done ({time.time()-t0:.1f}s)')

### STEP 10: Single-Target Grad-CAM Demo

Generate Grad-CAM for `food_score` (target 0) on the first image.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 10/17 — Single-Target Demo (food_score)')
print('='*60)

cam = compute_gradcam_for_image(
    model=model, sample=sample,
    target_idx=0, image_idx=0,
    target_layer=target_layer, device=device,
)

print(f'CAM shape    : {cam.shape}')
print(f'CAM range    : [{cam.min():.4f}, {cam.max():.4f}]')
assert cam.shape[0] > 1 and cam.shape[1] > 1, f'CAM has no spatial dims: {cam.shape}'
assert np.isfinite(cam).all(), 'CAM contains NaN/Inf'

overlay = overlay_cam_on_image(cam, sample['loaded_images'][0])

# Display: original | raw heatmap | overlay
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample['loaded_images'][0].resize((224, 224)))
axes[0].set_title('Original')
axes[1].imshow(cam, cmap='jet', vmin=0, vmax=1)
axes[1].set_title(f'Raw CAM ({cam.shape[0]}x{cam.shape[1]})')
axes[2].imshow(overlay)
axes[2].set_title('Grad-CAM: food_score')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'\nStep 10 done ({time.time()-t0:.1f}s)')

### STEP 11: All 5 Targets for One Image

Shows target specificity — different targets should produce different heatmaps.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 11/17 — All 5 Targets')
print('='*60)

cams_5target = []
for t_idx in range(NUM_TARGETS):
    c = compute_gradcam_for_image(
        model=model, sample=sample,
        target_idx=t_idx, image_idx=0,
        target_layer=target_layer, device=device,
    )
    cams_5target.append(c)
    print(f'  {FACTOR_NAMES[t_idx]:>10s}: range=[{c.min():.3f}, {c.max():.3f}]')

# Display: Original + 5 target overlays
fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
axes[0].imshow(sample['loaded_images'][0].resize((224, 224)))
axes[0].set_title('Original', fontsize=10, fontweight='bold')
axes[0].axis('off')

for t_idx in range(NUM_TARGETS):
    ov = overlay_cam_on_image(cams_5target[t_idx], sample['loaded_images'][0])
    axes[t_idx + 1].imshow(ov)
    axes[t_idx + 1].set_title(DISPLAY_NAMES[t_idx], fontsize=9, fontweight='bold')
    axes[t_idx + 1].axis('off')

plt.suptitle(f'Sample {demo_idx} — Grad-CAM: 5 Targets', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nStep 11 done ({time.time()-t0:.1f}s)')

### STEP 12: Full Sample Explanation with GradCAMExplainer

Uses the high-level `GradCAMExplainer` to generate all artifacts:
overlays, raw CAMs, comparison figure, and metadata.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 12/17 — Full Sample Explanation')
print('='*60)

explainer = GradCAMExplainer(model=model, device=device, output_dir=XAI_OUT_DIR)

sample_id = f'sample_{demo_idx:04d}'
results = explainer.explain_sample(sample=sample, sample_id=sample_id)

print(f'\nArtifacts generated for {sample_id}:')
print(f'  Comparison figure : {results["comparison_path"]}')
print(f'  Individual overlays: {len(results["individual_paths"])}')

# Display the comparison figure
comp_img = PILImage.open(results['comparison_path'])
fig, ax = plt.subplots(1, 1, figsize=(18, 4))
ax.imshow(comp_img)
ax.axis('off')
ax.set_title(f'{sample_id} — 5-Target Comparison', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nStep 12 done ({time.time()-t0:.1f}s)')

### STEP 13: Batch Processing

Process multiple samples and save all artifacts.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 13/17 — Batch Processing')
print('='*60)

batch_results = []

for sidx in SAMPLE_INDICES:
    ts = time.time()
    sample_id = f'sample_{sidx:04d}'
    print(f'\n--- Processing {sample_id} ---')

    try:
        s = load_single_sample(
            csv_path=SPLIT_CSV, idx=sidx,
            tokenizer=tokenizer, image_processor=image_processor,
            image_dir=IMAGE_DIR, device=device,
        )
        r = explainer.explain_sample(sample=s, sample_id=sample_id)

        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sample_id,
            'sample_idx': sidx,
            'num_images': s['num_real_images'],
            'status': 'success',
            'elapsed_s': round(elapsed, 1),
            'num_artifacts': len(r['individual_paths']),
        })
        print(f'  Done: {elapsed:.1f}s, {len(r["individual_paths"])} overlays')

    except Exception as e:
        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sample_id,
            'sample_idx': sidx,
            'status': 'failed',
            'error': str(e),
            'elapsed_s': round(elapsed, 1),
        })
        print(f'  FAILED: {e}')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save batch summary
batch_summary = {
    'phase': 'Phase 2: Grad-CAM',
    'experiment_id': EXP_ID,
    'split': SPLIT_NAME,
    'num_samples': len(SAMPLE_INDICES),
    'total_elapsed_s': round(time.time() - t0, 1),
    'results': batch_results,
}
summary_path = os.path.join(XAI_OUT_DIR, 'gradcam_batch_summary.json')
save_raw_values(batch_summary, summary_path)

print(f'\nBatch complete: {len(batch_results)} samples')
print(f'Step 13 done ({time.time()-t0:.1f}s)')

### STEP 14: Sanity Check — Target Specificity

Different targets should produce different heatmaps. Compute pairwise
Pearson correlation between 5 target heatmaps for one sample.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 14/17 — Target Specificity Check')
print('='*60)

# Use cams from step 11
flat_cams = [c.flatten() for c in cams_5target]
n = len(flat_cams)
corr_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if np.std(flat_cams[i]) > 1e-8 and np.std(flat_cams[j]) > 1e-8:
            corr_matrix[i, j] = np.corrcoef(flat_cams[i], flat_cams[j])[0, 1]
        else:
            corr_matrix[i, j] = 1.0 if i == j else 0.0

# Display correlation matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(FACTOR_NAMES, fontsize=9, rotation=45)
ax.set_yticklabels(FACTOR_NAMES, fontsize=9)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax)
ax.set_title(f'Grad-CAM Target Specificity — Sample {demo_idx}', fontsize=11)
plt.tight_layout()

corr_path = os.path.join(XAI_OUT_DIR, f'sample_{demo_idx:04d}', 'target_specificity_corr.png')
os.makedirs(os.path.dirname(corr_path), exist_ok=True)
fig.savefig(corr_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {corr_path}')

# Check: at least one pair with r < 0.9
off_diag = corr_matrix[np.triu_indices(n, k=1)]
min_corr = off_diag.min()
max_corr = off_diag.max()
mean_corr = off_diag.mean()

specificity_ok = min_corr < 0.95
print(f'Off-diagonal correlations: min={min_corr:.3f}, max={max_corr:.3f}, mean={mean_corr:.3f}')
if specificity_ok:
    print('Target specificity: PASSED (heatmaps differ across targets)')
else:
    print('WARNING: All target heatmaps are highly correlated. Possible head collapse.')

print(f'\nStep 14 done ({time.time()-t0:.1f}s)')

### STEP 15: Sanity Check — Reproducibility

Run Grad-CAM twice on the same sample. Results must be identical.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 15/17 — Reproducibility Check')
print('='*60)

cam_run1 = compute_gradcam_for_image(
    model=model, sample=sample,
    target_idx=0, image_idx=0,
    target_layer=target_layer, device=device,
)
cam_run2 = compute_gradcam_for_image(
    model=model, sample=sample,
    target_idx=0, image_idx=0,
    target_layer=target_layer, device=device,
)

is_identical = np.allclose(cam_run1, cam_run2, atol=1e-7)
max_diff = np.abs(cam_run1 - cam_run2).max()

print(f'Run 1 range  : [{cam_run1.min():.6f}, {cam_run1.max():.6f}]')
print(f'Run 2 range  : [{cam_run2.min():.6f}, {cam_run2.max():.6f}]')
print(f'Max diff     : {max_diff:.2e}')
print(f'Identical    : {is_identical}')

if is_identical:
    print('Reproducibility: PASSED')
else:
    print('WARNING: Results differ between runs. Check model determinism.')

print(f'\nStep 15 done ({time.time()-t0:.1f}s)')

### STEP 16: Gradient Flow Check

Verify that gradients actually flow from the target score to the target layer.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 16/17 — Gradient Flow Check')
print('='*60)

gradient_ok = True

for t_idx in range(NUM_TARGETS):
    gradients = {}

    def bwd_hook(module, grad_input, grad_output):
        gradients['value'] = grad_output[0].detach()

    handle = target_layer.register_full_backward_hook(bwd_hook)
    model.zero_grad()

    try:
        with torch.enable_grad():
            pv = sample['pixel_values'].clone().detach().requires_grad_(True)
            output = model(
                input_ids=sample['input_ids'],
                attention_mask=sample['attention_mask'],
                pixel_values=pv,
                num_images=sample.get('num_images'),
            )
            preds = output[0] if isinstance(output, tuple) else output
            preds[0, t_idx].backward(retain_graph=False)

        if 'value' in gradients:
            grad_sum = gradients['value'].abs().sum().item()
            status = 'OK' if grad_sum > 0 else 'ZERO'
            if grad_sum == 0:
                gradient_ok = False
            print(f'  {FACTOR_NAMES[t_idx]:>10s}: grad_abs_sum={grad_sum:.4f} [{status}]')
        else:
            print(f'  {FACTOR_NAMES[t_idx]:>10s}: NO GRADIENT CAPTURED')
            gradient_ok = False
    finally:
        handle.remove()
        model.zero_grad()
        gradients.clear()

if gradient_ok:
    print('\nGradient flow: PASSED (all targets have non-zero gradients)')
else:
    print('\nWARNING: Some targets have zero gradients. Grad-CAM may be uninformative.')

print(f'\nStep 16 done ({time.time()-t0:.1f}s)')

### STEP 17: Final Summary

In [ ]:
print('='*60)
print('  PHASE 2 GRAD-CAM — FINAL SUMMARY')
print('='*60)

# Count artifacts
artifact_count = 0
for root, dirs, files in os.walk(XAI_OUT_DIR):
    artifact_count += len(files)

print(f'  Experiment     : {EXP_ID}')
print(f'  Split          : {SPLIT_NAME}')
print(f'  Samples        : {len(SAMPLE_INDICES)}')
print(f'  Total artifacts: {artifact_count}')
print(f'  Output dir     : {XAI_OUT_DIR}')
print()

checks = [
    ('Model loaded',          True),
    ('Target layer found',    target_layer is not None),
    ('Single CAM generated',  cam is not None and cam.shape[0] > 1),
    ('5-target comparison',   len(cams_5target) == 5),
    ('Target specificity',    specificity_ok),
    ('Reproducibility',       is_identical),
    ('Gradient flow',         gradient_ok),
    ('Batch processing',      len(batch_results) == len(SAMPLE_INDICES)),
]

all_passed = True
for desc, passed in checks:
    status = 'PASSED' if passed else 'FAILED'
    if not passed:
        all_passed = False
    print(f'  [{status:6s}] {desc}')

print('='*60)
if all_passed:
    print('  All checks PASSED. Phase 2 Grad-CAM is complete.')
else:
    print('  Some checks FAILED. Review the output above.')
print('='*60)